# 01. Quality Gate 1: евристичний бейзлайн, ворота конфігурації, адаптивний імпутер

Ноутбук демонструє артефакти Quality Gate 1:
1. **Non-ML Heuristic Benchmark** — Accuracy / Recall / Precision на всьому наборі, без навчання і без розбиття (п. 5.1).
2. **Спрацювання перевірки конфігурації** на навмисно зіпсованій конфігурації (п. 3.4.9, крок 2).
3. **Збирання конвеєра** з двох конфігурацій і протокол верифікації (п. 3.4.9, кроки 5–7).
4. **Аудиторський слід** адаптивного імпутера, навченого лише на тренувальній підвибірці (п. 3.4.9, крок 4; джерело таблиці п. 5.3).

Класифікатори **не навчаються**.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dynamic_pipeline_builder.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
FIG = ROOT / "artifacts" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c9c8c3", "axes.labelcolor": "#52514e", "text.color": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

import json
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from src.dynamic_pipeline_builder import DynamicPipelineBuilder

ART = ROOT / "artifacts"
LR, RF = ROOT / "config/config_lr.yaml", ROOT / "config/config_rf.yaml"

## 1. Евристичний бейзлайн (Non-ML Benchmark)

**Правило** (сформульоване з доменної логіки кредитного офіцера, без підбору порогів під дані):

> **Bad**, якщо на поточному рахунку від'ємний залишок (`checking_account == "<0 DM"`)
> **АБО** кредит довший за 24 місяці (`duration > 24`) за заощаджень менше 100 DM (`savings_account == "<100 DM"`).

Позитивний клас — **bad** (подія ризику). Рахується на розкодованому наборі **без** штучних пропусків — евристика не має параметрів, тож перенавчатися їй нема на чому.

In [2]:
df = pd.read_csv(ROOT / "data/processed/german_credit_clean.csv")
y_true = df["bad_credit"]
rule = (df["checking_account"] == "<0 DM") | ((df["duration"] > 24) & (df["savings_account"] == "<100 DM"))
y_heur = rule.astype(int)

def metrics(y, p):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y, p),
        "recall_bad": recall_score(y, p, pos_label=1, zero_division=0),
        "precision_bad": precision_score(y, p, pos_label=1, zero_division=0),
        "cost_per_applicant": (5 * fn + 1 * fp) / len(y),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    }

bench = pd.DataFrame({
    "heuristic": metrics(y_true, y_heur),
    "trivial: all good": metrics(y_true, np.zeros_like(y_true)),
    "trivial: all bad": metrics(y_true, np.ones_like(y_true)),
}).T
bench.round(3)

,accuracy,recall_bad,precision_bad,cost_per_applicant,TP,FP,FN,TN
heuristic,0.693,0.58,0.49,0.811,174.0,181.0,126.0,519.0
trivial: all good,0.700,0.00,0.00,1.500,0.0,0.0,300.0,700.0
trivial: all bad,0.300,1.00,0.30,0.700,300.0,700.0,0.0,0.0


In [3]:
print("Accuracy: ", round(accuracy_score(y_true, y_heur), 3))
print("Recall:   ", round(recall_score(y_true, y_heur, pos_label=1), 3))
print("Precision:", round(precision_score(y_true, y_heur, pos_label=1), 3))
_ = (ART / "heuristic_benchmark.json").write_text(
    json.dumps({k: {m: float(v) for m, v in row.items()} for k, row in bench.iterrows()}, indent=2), encoding="utf-8")

Accuracy:  0.693
Recall:    0.58
Precision: 0.49


**Інтерпретація.** Accuracy евристики нижча за тривіальне «усі good» (0.70), хоча евристика знаходить 58% поганих кредитів, а тривіальне правило — жодного. Тобто Accuracy на незбалансованих класах (30/70) не відображає цінності для бізнесу, і ML Metric формулюється через Recall/Precision на класі bad та вартість помилок.

## 2. Ворота якості: навмисно зіпсована конфігурація

`tests/fixtures/config_protected_leak.yaml` — копія `config_lr.yaml`, у якій захищену ознаку `sex` внесено до `features.categorical_nominal`. Перевірка викликається **в конструкторі**, до збирання `ColumnTransformer`, тому об'єкт навіть не створюється.

In [4]:
import difflib
broken = ROOT / "tests/fixtures/config_protected_leak.yaml"
for line in difflib.unified_diff(LR.read_text(encoding="utf-8").splitlines(),
                                 broken.read_text(encoding="utf-8").splitlines(),
                                 "config_lr.yaml", broken.name, lineterm="", n=0):
    print(line)

--- config_lr.yaml
+++ config_protected_leak.yaml
@@ -0,0 +1,4 @@
+# НАВМИСНО ЗІПСОВАНА конфігурація для демонстрації воріт якості (п. 3.4.9, крок 2).
+# Відрізняється від config_lr.yaml рядком '    - sex' у categorical_nominal.
+# Очікувано: DynamicPipelineBuilder(...) піднімає ValueError у конструкторі.
+
@@ -24,0 +29 @@
+    - sex                       # <-- ПОРУШЕННЯ: захищена ознака у features


In [5]:
builder = None
try:
    builder = DynamicPipelineBuilder(str(broken))
except ValueError as err:
    print(f"ValueError: {err}")
print("Об'єкт створено:", builder is not None)

ValueError: Захищені ознаки потрапили у features: {'sex'}
Об'єкт створено: False


Друга межа захисту — структурна перевірка `validate_schema`. Наприклад, ціль у `features` (прямий витік цілі) також зупиняє конструктор:

In [6]:
import tempfile, copy
cfg = yaml.safe_load(LR.read_text(encoding="utf-8"))
bad_cfg = copy.deepcopy(cfg)
bad_cfg["features"]["numeric_continuous"].append(cfg["target"])
with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False, encoding="utf-8") as f:
    yaml.safe_dump(bad_cfg, f, allow_unicode=True)
try:
    DynamicPipelineBuilder(f.name)
except ValueError as err:
    print(f"ValueError: {err}")

ValueError: Цільова змінна 'bad_credit' внесена до features — прямий витік цілі


## 3. Дві конфігурації — один рядок різниці

In [7]:
for line in difflib.unified_diff(LR.read_text(encoding="utf-8").splitlines(),
                                 RF.read_text(encoding="utf-8").splitlines(),
                                 LR.name, RF.name, lineterm="", n=0):
    print(line)

--- config_lr.yaml
+++ config_rf.yaml
@@ -6 +6 @@
-model_family: additive          # additive | partitioning — яку гілку transform_policy читати
+model_family: partitioning       # additive | partitioning — яку гілку transform_policy читати


In [8]:
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError

def describe(ct):
    return {group: " → ".join(name for name, _ in pipe.steps) for group, pipe, _ in ct.transformers}

ct_lr, ct_rf = DynamicPipelineBuilder(str(LR)).build(), DynamicPipelineBuilder(str(RF)).build()
display(pd.DataFrame({"additive (config_lr)": describe(ct_lr), "partitioning (config_rf)": describe(ct_rf)}))
try:
    check_is_fitted(ct_lr)
except NotFittedError:
    print("build() повернув НЕНАВЧЕНИЙ ColumnTransformer — до даних він не звертався.")

,additive (config_lr),partitioning (config_rf)
numeric_continuous,impute → log1p → standard_scaler,impute → winsorize
numeric_discrete,impute → standard_scaler,impute
categorical_nominal,impute → woe,impute → one_hot
categorical_ordinal,impute → ordinal_encoder → standard_scaler,impute → ordinal_encoder


build() повернув НЕНАВЧЕНИЙ ColumnTransformer — до даних він не звертався.


## 4. Протокол верифікації (`run_verification.py`)

In [9]:
import subprocess
res = subprocess.run([sys.executable, "run_verification.py"], cwd=ROOT, capture_output=True, text=True)
print(res.stdout, res.stderr, "exit code:", res.returncode)

config/config_lr.yaml: перевірки пройдено
config/config_rf.yaml: перевірки пройдено
  exit code: 0


## 5. Навчання препроцесора на тренувальній підвибірці й аудиторський слід

Щоб зафіксувати, *що побачив імпутер*, його треба навчити — а навчати можна **лише після розбиття**. Тому тут виконується демонстраційне розбиття 80/20 зі стратифікацією за ціллю (`random_state=42`). Це **єдиний стохастичний крок** ЛР №1, і він не належить до конвеєра: сам конвеєр детермінований. Офіційне розбиття для моделювання з'явиться в ЛР №2.

Класифікатор не навчається: `fit` викликається лише на `ColumnTransformer`. Ціль передається в `fit`, бо її потребує WoE-кодувальник; `AdaptiveImputer` її ігнорує.

In [10]:
from sklearn.model_selection import train_test_split

data = pd.read_csv(ROOT / "data/processed/german_credit_mcar.csv")
y = data.pop("bad_credit")
X_train, X_test, y_train, y_test = train_test_split(data, y, test_size=0.2, stratify=y, random_state=42)
print("train:", X_train.shape, " test:", X_test.shape)

ct_lr.fit(X_train, y_train)
ct_rf.fit(X_train, y_train)
audit_lr = DynamicPipelineBuilder.audit_trail(ct_lr)
audit_rf = DynamicPipelineBuilder.audit_trail(ct_rf)
audit_lr.to_csv(ART / "audit_trail_lr.csv", index=False)
audit_rf.to_csv(ART / "audit_trail_rf.csv", index=False)
audit_lr

train: (800, 21)  test: (200, 21)


,feature,group,missing_share,max_share,strategy,fill_value,add_indicator,transforms
0,duration,numeric_continuous,0.0000,0.1,median,18.0,False,log1p → standard_scaler
1,credit_amount,numeric_continuous,0.2025,0.4,median,2317.0,True,log1p → standard_scaler
2,age,numeric_continuous,0.0525,0.1,median,33.0,False,log1p → standard_scaler
3,installment_rate,numeric_discrete,0.0000,0.1,median,3.0,False,standard_scaler
4,residence_since,numeric_discrete,0.5012,1.0,constant,0,True,standard_scaler
5,existing_credits,numeric_discrete,0.0000,0.1,median,1.0,False,standard_scaler
6,checking_account,categorical_nominal,0.0000,0.1,most_frequent,no account,False,woe
7,credit_history,categorical_nominal,0.0000,0.1,most_frequent,existing paid duly,False,woe
8,purpose,categorical_nominal,0.2500,1.0,constant,Missing,False,woe
9,savings_account,categorical_nominal,0.0650,0.1,most_frequent,<100 DM,False,woe


Сирий аудиторський слід одного імпутера (атрибут `audit_`) — те, що показується на захисті:

In [11]:
imputer = ct_lr.named_transformers_["numeric_continuous"].named_steps["impute"]
for col, entry in imputer.audit_.items():
    print(f"{col:15s} {entry}")

duration        {'missing_share': 0.0, 'rule': {'max_share': 0.1, 'strategy': 'median'}, 'fill_value': 18.0, 'add_indicator': False}
credit_amount   {'missing_share': 0.2025, 'rule': {'max_share': 0.4, 'strategy': 'median', 'add_indicator': True}, 'fill_value': 2317.0, 'add_indicator': True}
age             {'missing_share': 0.0525, 'rule': {'max_share': 0.1, 'strategy': 'median'}, 'fill_value': 33.0, 'add_indicator': False}


### Таблиця метаданих (джерело п. 5.3): стратегія A — `additive`, стратегія Б — `partitioning`

In [12]:
meta = audit_lr[["feature", "group", "missing_share", "max_share", "strategy", "add_indicator"]].copy()
meta["rule"] = meta.apply(lambda r: f"max_share: {r.max_share:.2f}, {r.strategy}" + (" +I" if r.add_indicator else ""), axis=1)
meta["strategy_A_additive"] = audit_lr["transforms"]
meta["strategy_B_partitioning"] = audit_rf["transforms"]
meta = meta[["feature", "group", "missing_share", "rule", "strategy_A_additive", "strategy_B_partitioning"]]
meta.to_csv(ART / "metadata_table.csv", index=False)
meta

,feature,group,missing_share,rule,strategy_A_additive,strategy_B_partitioning
0,duration,numeric_continuous,0.0000,"max_share: 0.10, median",log1p → standard_scaler,winsorize
1,credit_amount,numeric_continuous,0.2025,"max_share: 0.40, median +I",log1p → standard_scaler,winsorize
2,age,numeric_continuous,0.0525,"max_share: 0.10, median",log1p → standard_scaler,winsorize
3,installment_rate,numeric_discrete,0.0000,"max_share: 0.10, median",standard_scaler,(лише імпутація)
4,residence_since,numeric_discrete,0.5012,"max_share: 1.00, constant +I",standard_scaler,(лише імпутація)
5,existing_credits,numeric_discrete,0.0000,"max_share: 0.10, median",standard_scaler,(лише імпутація)
6,checking_account,categorical_nominal,0.0000,"max_share: 0.10, most_frequent",woe,one_hot
7,credit_history,categorical_nominal,0.0000,"max_share: 0.10, most_frequent",woe,one_hot
8,purpose,categorical_nominal,0.2500,"max_share: 1.00, constant",woe,one_hot
9,savings_account,categorical_nominal,0.0650,"max_share: 0.10, most_frequent",woe,one_hot


## 6. Перетворення тестової підвибірки

In [13]:
Z_lr_train, Z_lr_test = ct_lr.transform(X_train), ct_lr.transform(X_test)
Z_rf_train, Z_rf_test = ct_rf.transform(X_train), ct_rf.transform(X_test)
pd.DataFrame({
    "train shape": [Z_lr_train.shape, Z_rf_train.shape],
    "test shape": [Z_lr_test.shape, Z_rf_test.shape],
    "NaN у виході (test)": [int(np.isnan(Z_lr_test).sum()), int(np.isnan(Z_rf_test).sum())],
}, index=["additive", "partitioning"])

,train shape,test shape,NaN у виході (test)
additive,"(800, 19)","(200, 19)",0
partitioning,"(800, 50)","(200, 50)",0


In [14]:
pd.DataFrame(Z_lr_test, columns=ct_lr.get_feature_names_out(), index=X_test.index).head().round(3)

,numeric_continuous__duration,numeric_continuous__credit_amount,numeric_continuous__age,numeric_continuous__credit_amount_missing,numeric_discrete__installment_rate,numeric_discrete__residence_since,numeric_discrete__existing_credits,numeric_discrete__residence_since_missing,categorical_nominal__checking_account,categorical_nominal__credit_history,categorical_nominal__purpose,categorical_nominal__savings_account,categorical_nominal__other_debtors,categorical_nominal__property,categorical_nominal__other_installment_plans,categorical_nominal__housing,categorical_nominal__telephone,categorical_ordinal__employment_since,categorical_ordinal__job
30,0.012,-0.039,0.222,1.985,0.052,-0.870,-0.719,0.998,-0.408,-0.099,-0.154,-0.249,0.013,0.508,-0.666,0.202,0.111,1.545,0.155
128,-0.688,-0.365,0.020,-0.504,0.942,0.369,1.045,-1.003,-0.408,0.700,0.856,-0.249,0.013,-0.032,0.151,0.202,0.111,-1.948,1.740
289,0.519,-0.039,1.241,1.985,0.942,1.608,-0.719,-1.003,-0.761,-0.096,0.374,-0.249,0.013,0.508,-0.279,0.202,-0.074,-1.250,0.155
216,0.012,0.395,-0.305,-0.504,0.052,-0.250,-0.719,-1.003,-0.761,-1.286,0.056,-0.249,0.013,0.005,-0.666,0.202,0.111,0.148,0.155
966,0.729,0.086,-0.085,-0.504,0.942,-0.870,1.045,0.998,-0.408,0.700,0.374,0.720,0.013,0.005,0.151,0.202,-0.074,-0.551,-1.431


WoE-кодування (оцінене лише на train): значення для `checking_account` і Information Value номінальних ознак.

In [15]:
woe = ct_lr.named_transformers_["categorical_nominal"].named_steps["woe"]
display(pd.Series(woe.mapping_["checking_account"], name="WoE").round(3).to_frame())
pd.Series(woe.iv_, name="IV (train)").sort_values(ascending=False).round(3).to_frame()

,WoE
0-200 DM,-0.408
<0 DM,-0.761
>=200 DM,0.205
no account,1.136


,IV (train)
checking_account,0.609
credit_history,0.262
savings_account,0.172
property,0.145
purpose,0.143
other_installment_plans,0.091
housing,0.089
other_debtors,0.025
telephone,0.008


## 7. Метод `transform` нічого не переобчислює (контрольне питання 28)

Подвоюємо частку пропусків `credit_amount` у тестовій підвибірці. Правило (`median +I`) і значення заповнення (медіана train) мають залишитися тими самими, розмірність виходу — теж.

In [16]:
X_shift = X_test.copy()
observed = X_shift.index[X_shift["credit_amount"].notna()]
target_missing = 2 * audit_lr.set_index("feature").loc["credit_amount", "missing_share"]
k = int(round(target_missing * len(X_shift))) - int(X_shift["credit_amount"].isna().sum())
X_shift.loc[observed[:k], "credit_amount"] = np.nan
print(f"частка пропусків credit_amount: train={audit_lr.set_index('feature').loc['credit_amount','missing_share']:.3f}, "
      f"test зі зсувом={X_shift['credit_amount'].isna().mean():.3f}")

audit_before = {c: dict(v) for c, v in imputer.audit_.items()}
Z_shift = ct_lr.transform(X_shift)
print("аудиторський слід не змінився:", imputer.audit_ == audit_before)
print("розмірність виходу та сама:", Z_shift.shape == Z_lr_test.shape)
print("значення заповнення (медіана train):", imputer.fill_values_["credit_amount"])

частка пропусків credit_amount: train=0.203, test зі зсувом=0.405
аудиторський слід не змінився: True
розмірність виходу та сама: True
значення заповнення (медіана train): 2317.0


## 8. Захищені ознаки залишаються в даних, але не в моделі

In [17]:
b = DynamicPipelineBuilder(str(LR))
out_names = set(ct_lr.get_feature_names_out()) | set(ct_rf.get_feature_names_out())
print("protected_attributes:", b.protected_attributes)
print("у виході конвеєра:", sorted(n for n in out_names if any(n.split("__", 1)[1].startswith(a) for a in b.protected_attributes)))
X_test[b.protected_attributes].head()

protected_attributes: ['sex', 'personal_status_sex', 'foreign_worker']
у виході конвеєра: []


,sex,personal_status_sex,foreign_worker
30,male,male: married/widowed,yes
128,male,male: single,yes
289,male,male: married/widowed,yes
216,male,male: single,yes
966,male,male: single,yes
